# Module 4 — RAG Pipeline (Qdrant + Groq)
Mental Health Support Chatbot**

Retrieval-Augmented Generation pipeline:

1. **Embed** the mental health counseling dataset with `sentence-transformers`
2. **Store** vectors in free cloud Qdrant
3. **Retrieve** top-k relevant Q&A pairs for any user query
4. **Generate** an empathetic answer using Groq (`meta-llama/llama-4-maverick-17b-128e-instruct` / gpt-oss-120b equivalent)

## 1. Install

In [ ]:
!pip install -q qdrant-client sentence-transformers groq datasets pandas

## 2. Imports

In [1]:
import os, json
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from groq import Groq


e:\final_final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. API keys & config

In [ ]:
QDRANT_URL     = "ADD_YOUR_URL"  
QDRANT_API_KEY = "ADD_YOUR_API_KEY"
GROQ_API_KEY   = "ADD_YOUR_API_KEY"

COLLECTION_NAME = "mental_health_qa"
EMBED_MODEL     = "all-MiniLM-L6-v2"         
LLM_MODEL       = "openai/gpt-oss-120b"

groq_client   = Groq(api_key=GROQ_API_KEY)
qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,  
)

## 4. Load the Mental Health Counseling dataset

In [4]:
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")
df = dataset.to_pandas()[["Context", "Response"]].dropna().drop_duplicates().reset_index(drop=True)

print(f"Rows: {len(df)}")
print(df.head(3))

Rows: 2752
                                             Context  \
0  I'm going through some things with my feelings...   
1  I'm going through some things with my feelings...   
2  I'm going through some things with my feelings...   

                                            Response  
0  If everyone thinks you're worthless, then mayb...  
1  Hello, and thank you for your question and see...  
2  First thing I'd suggest is getting the sleep y...  


## 5. Embed contexts

In [5]:
embedder = SentenceTransformer(EMBED_MODEL)

print("Embedding contexts...")
embeddings = embedder.encode(
    df["Context"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f"Embeddings shape: {embeddings.shape}")
VECTOR_DIM = embeddings.shape[1]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1616.47it/s]


Embedding contexts...


Batches: 100%|██████████| 43/43 [00:55<00:00,  1.28s/it]

Embeddings shape: (2752, 384)


## 6. Upload to Qdrant

In [ ]:
import time

qdrant_client.delete_collection(COLLECTION_NAME)

existing = [c.name for c in qdrant_client.get_collections().collections]

if COLLECTION_NAME in existing:
    count = qdrant_client.count(COLLECTION_NAME).count
    print(f"Collection '{COLLECTION_NAME}' already exists with {count} vectors — skipping upload.")
else:
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
    )

    points = [
        PointStruct(
            id=int(i),
            vector=embeddings[i].tolist(),
            payload={"context": df.loc[i, "Context"], "response": df.loc[i, "Response"]},
        )
        for i in range(len(df))
    ]

    BATCH_SIZE = 50  
    MAX_RETRIES = 3
    total = len(points)

    for start in range(0, total, BATCH_SIZE):
        batch = points[start : start + BATCH_SIZE]
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                qdrant_client.upsert(collection_name=COLLECTION_NAME, points=batch)
                print(f"  Uploaded {min(start + BATCH_SIZE, total)}/{total}", end="\r")
                break
            except Exception as e:
                if attempt == MAX_RETRIES:
                    raise
                print(f"  Batch {start} failed (attempt {attempt}): {e} — retrying in 3s...")
                time.sleep(3)

    print(f"\nDone. Uploaded {total} vectors to '{COLLECTION_NAME}'.")


  Uploaded 2752/2752
Done. Uploaded 2752 vectors to 'mental_health_qa'.


## 7. Retrieval function

In [11]:
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Embed the query and return top_k matching Q&A pairs from Qdrant."""
    from qdrant_client.models import NamedVector
    query_vector = embedder.encode(query).tolist()
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
    ).points
    return [
        {
            "context":  hit.payload["context"],
            "response": hit.payload["response"],
            "score":    round(hit.score, 3),
        }
        for hit in results
    ]

hits = retrieve("I feel very anxious and cannot sleep")
for h in hits:
    print(f"[score={h['score']}]", h["context"][:80])

[score=0.722] I've been having horrible anxiety for the last week. I can't sleep. I get a sens
[score=0.645] I have a lot of issues going on right now. First of all, I have a lot of trouble
[score=0.645] I have a lot of issues going on right now. First of all, I have a lot of trouble


## 8. Generation function

In [12]:
RAG_SYSTEM_PROMPT = """You are an empathetic mental health support assistant.
You are given retrieved counseling examples to help you answer.
Give a warm, supportive, and grounded response.
Do NOT diagnose. Do NOT prescribe medication. Encourage professional help when needed.
Be concise — 3 to 5 sentences.
"""

def generate_answer(user_query: str, retrieved_docs: list[dict], emotion: str = "neutral", language: str = "English") -> str:
    """Generate a RAG-based answer using Groq."""
    context_block = "\n\n".join(
        f"Example {i+1}:\nUser: {d['context']}\nCounselor: {d['response']}"
        for i, d in enumerate(retrieved_docs)
    )

    user_message = f"""Detected emotion: {emotion}
User language: {language}
User message: {user_query}

Relevant counseling examples:
{context_block}

Please respond in {language}."""

    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.7,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()

## 9. Full RAG pipeline — end-to-end test

In [13]:
def rag_answer(user_query: str, emotion: str = "neutral", language: str = "English") -> str:
    """One-call wrapper used by app.py."""
    docs = retrieve(user_query, top_k=3)
    return generate_answer(user_query, docs, emotion=emotion, language=language)


queries = [
    ("I have been feeling very anxious and I cannot sleep at night", "fear",    "English"),
    ("أنا أشعر بالاكتئاب الشديد ولا أعرف ماذا أفعل",             "sadness", "Arabic"),
]

for q, emo, lang in queries:
    print(f"\n{'='*60}")
    print(f"Query [{lang}, {emo}]: {q}")
    print("-"*60)
    answer = rag_answer(q, emotion=emo, language=lang)
    print(answer)


Query [English, fear]: I have been feeling very anxious and I cannot sleep at night
------------------------------------------------------------
I’m really sorry you’ve been dealing with so much anxiety and sleeplessness—it can feel overwhelming when fear keeps you up at night. Trying a simple bedtime routine—like dimming lights, limiting screens, and doing a few slow, deep breaths or gentle stretches—can help signal your body that it’s time to rest. If racing thoughts keep returning, jotting them down on paper for a few minutes before bed can give them a place to settle. Remember, reaching out to a healthcare professional or therapist can provide personalized strategies and support. You deserve help and relief, and taking that first step is a brave and important move.

Query [Arabic, sadness]: أنا أشعر بالاكتئاب الشديد ولا أعرف ماذا أفعل
------------------------------------------------------------
أشعر بألمك وأتفهم أن الشعور بالاكتئاب يمكن أن يكون ثقيلًا جدًا. حاول أن تتحدث مع شخص تث

## 10. Save RAG config

In [14]:
config = {
    "qdrant_url":        QDRANT_URL,
    "collection_name":   COLLECTION_NAME,
    "embed_model":       EMBED_MODEL,
    "llm_model":         LLM_MODEL,
    "top_k":             3,
}

with open("rag_config.json", "w") as f:
    json.dump(config, f, indent=2)